In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
%pip install \
opentelemetry-sdk==1.38.0 \
opentelemetry-proto==1.38.0 \
opentelemetry-exporter-otlp-proto-common==1.38.0 \
langchain \
langchain-community \
langchain-core \
langchain-aws \
langgraph \
chromadb \
sentence-transformers \
transformers \
openai-whisper \
pillow \
tqdm \
requests==2.32.4 \
accelerate \
pydantic \
boto3 \
botocore

In [ ]:
import json
from tqdm import tqdm
import os
import torch
import numpy as np
import whisper
import requests
import chromadb
from sentence_transformers import SentenceTransformer
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
from io import BytesIO
from tqdm import tqdm
import torch.nn.functional as F
import pickle
import subprocess
from collections import defaultdict
from typing import Optional,List,TypedDict,Any
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.retrievers import BaseRetriever
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from pydantic import BaseModel, Field
import time

In [ ]:
DRIVE_BASE     = "/content/drive/MyDrive/Dataset-json"
INPUT_JSON     = f"{DRIVE_BASE}/aws_input.json"
PROCESSED_JSON = f"{DRIVE_BASE}/processed.json"           # was: local "processed.json"
THUMB_DIR      = f"{DRIVE_BASE}/thumbnails"               # was: local "thumbnails/"
CHROMA_DIR     = f"{DRIVE_BASE}/chroma_db"                # was: local "./chroma_db"

os.makedirs(THUMB_DIR,  exist_ok=True)
os.makedirs(CHROMA_DIR, exist_ok=True)

HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}

# ── Device ────────────────────────────────────────────────────────────────────
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device.upper()}")


Device: CPU


In [ ]:

# ── Normalize ─────────────────────────────────────────────────────────────────
def normalize_instagram(r: dict) -> dict:
    caption = r.get("caption", "")
    images  = r.get("images", [])
    return {
        "id":         f"ig_{r.get('id', '')}",
        "source":     "instagram",
        "title":      caption[:100],
        "caption":    caption,
        "video_url":  r.get("videoUrl", ""),
        "thumbnail":  images[0] if images else "",
        "likes":      r.get("likesCount", 0),
        "comments":   r.get("commentsCount", 0),
        "duration":   r.get("videoDuration", 0),
        "owner":      r.get("ownerUsername", ""),
        "hashtags":   r.get("hashtags", []),
        "timestamp":  r.get("timestamp", ""),
        "transcript": "",
        "thumb_path": None,
        "rich_text":  "",
    }

# ── Load + Deduplicate ────────────────────────────────────────────────────────
print("📂 Loading JSON...")
try:
    with open(INPUT_JSON, "r", encoding="utf-8") as f:
        ig_data = json.load(f)
except FileNotFoundError:
    raise FileNotFoundError(f"Input file not found: {INPUT_JSON}")
except json.JSONDecodeError as e:
    raise ValueError(f"Malformed JSON in input file: {e}")

seen    = set()
deduped = []
for r in tqdm(ig_data, desc="📸 Processing Instagram"):
    norm = normalize_instagram(r)
    key  = norm["video_url"] or norm["id"]
    if not key:
        # Both fields empty — skip instead of collapsing all such records to one key
        tqdm.write(f"⚠️  Skipping record with no video_url or id: {r}")
        continue
    if key not in seen:
        deduped.append(norm)
        seen.add(key)

print(f"✅ {len(deduped)} unique reels loaded")

# ── ffmpeg helpers (defined BEFORE transcribe to avoid NameError) ─────────────
def valid_video(path: str) -> bool:
    try:
        subprocess.run(                                    # subprocess now imported
            ["ffprobe", "-v", "error", path],
            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True
        )
        return True
    except subprocess.CalledProcessError:
        return False

def extract_audio(video_path: str, audio_path: str) -> bool:
    result = subprocess.run([
        "ffmpeg", "-y", "-i", video_path,
        "-vn", "-acodec", "pcm_s16le", "-ar", "16000", "-ac", "1", audio_path
    ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    return result.returncode == 0                         # was: no return code check

# ── Models ────────────────────────────────────────────────────────────────────
CLIP_MODEL_ID = "openai/clip-vit-base-patch32"
print("Loading models...")
text_model     = SentenceTransformer("all-MiniLM-L6-v2", device=device)
clip_model     = CLIPModel.from_pretrained(CLIP_MODEL_ID).to(device).eval()  # chained .eval()
clip_processor = CLIPProcessor.from_pretrained(CLIP_MODEL_ID)
whisper_model  = whisper.load_model("medium", device=device)
print("✅ All models loaded!")

# ── Helper functions ──────────────────────────────────────────────────────────
def transcribe(video_url: str, reel_id: str) -> str:
    if not video_url:
        return ""
    tmp_video = f"tmp_{reel_id}.mp4"
    tmp_audio = f"tmp_{reel_id}.wav"
    try:
        r = requests.get(video_url, timeout=30, stream=True, headers=HEADERS)
        if r.status_code != 200:
            raise Exception(f"HTTP {r.status_code}")
        size = 0
        with open(tmp_video, "wb") as f:
            for chunk in r.iter_content(chunk_size=16384):
                if chunk:
                    f.write(chunk)
                    size += len(chunk)
        if size < 500_000:
            raise Exception("Downloaded file too small")
        if not valid_video(tmp_video):
            raise Exception("Invalid video file")
        if not extract_audio(tmp_video, tmp_audio):       # now checks return value
            raise Exception("ffmpeg audio extraction failed")
        result = whisper_model.transcribe(
            tmp_audio, fp16=(device == "cuda"),
            language="en", verbose=False, condition_on_previous_text=False,
        )
        return result["text"].strip()
    except Exception as e:
        tqdm.write(f"⚠️  Whisper failed {reel_id}: {e}")
        return ""
    finally:
        for f in [tmp_video, tmp_audio]:
            if os.path.exists(f):
                os.remove(f)

def download_thumbnail(url: str, save_path: str) -> bool:
    try:
        r = requests.get(url, timeout=10, headers=HEADERS)
        if r.status_code != 200:                          # was: no status check
            return False
        img = Image.open(BytesIO(r.content)).convert("RGB")
        if img.size[0] < 50 or img.size[1] < 50:
            return False
        img.save(save_path, "JPEG", quality=85)
        return True
    except Exception:
        return False

def get_clip_embedding(image_path: str):
    try:
        image  = Image.open(image_path).convert("RGB")
        inputs = clip_processor(images=image, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            features = clip_model.get_image_features(**inputs)
            features = features / features.norm(dim=-1, keepdim=True)
        return features[0].cpu().numpy().tolist()
    except Exception as e:
        tqdm.write(f"⚠️  CLIP embedding failed {image_path}: {e}")  # was: silent None
        return None

# ── ChromaDB ──────────────────────────────────────────────────────────────────
client           = chromadb.PersistentClient(path=CHROMA_DIR)
text_collection  = client.get_or_create_collection("reel_text",   metadata={"hnsw:space": "cosine"})
image_collection = client.get_or_create_collection("reel_images", metadata={"hnsw:space": "cosine"})
print(f"Text: {text_collection.count()} | Image: {image_collection.count()}")

# ── Checkpoint: pre-load existing IDs to enable resume ────────────────────────
existing_text_ids  = set(text_collection.get(include=[])["ids"])
existing_image_ids = set(image_collection.get(include=[])["ids"])

# ── Main processing loop ──────────────────────────────────────────────────────
text_ok = text_fail = img_ok = img_fail = skipped = 0

for reel in tqdm(deduped, desc="Processing reels", unit="reel"):
    reel_id   = reel["id"]                               # direct access — keys guaranteed by normalize_instagram
    video_url = reel["video_url"]
    caption   = reel["caption"]

    already_has_text  = reel_id in existing_text_ids
    already_has_image = reel_id in existing_image_ids

    if already_has_text and already_has_image:            # resume: skip fully-indexed reels
        skipped += 1
        continue

    # Step 1: transcript
    transcript = reel["transcript"].strip()
    if not transcript and video_url:
        transcript = transcribe(video_url, reel_id)
        reel["transcript"] = transcript

    # Step 2: thumbnail
    thumb_path   = reel["thumb_path"]
    thumb_exists = bool(thumb_path and os.path.exists(thumb_path))
    if not thumb_exists:
        thumb_url = reel["thumbnail"]
        if thumb_url:
            save_path = f"{THUMB_DIR}/{reel_id}.jpg"
            if download_thumbnail(thumb_url, save_path):
                thumb_path         = save_path
                reel["thumb_path"] = thumb_path
                thumb_exists       = True

    metadata = {
        "video_url": video_url,
        "owner":     reel["owner"],                      # direct access throughout
        "likes":     reel["likes"],
        "comments":  reel["comments"],
        "duration":  reel["duration"],
        "source":    reel["source"],
        "caption":   caption[:500],
    }

    # Step 3: text embedding (skip if already indexed)
    if not already_has_text:
        full_text = f"{caption} {transcript}".strip()
        if full_text:
            try:
                text_embed = text_model.encode(full_text, normalize_embeddings=True).tolist()
                text_collection.upsert(
                    ids=[reel_id],
                    embeddings=[text_embed],
                    documents=[full_text[:2000]],
                    metadatas=[metadata]
                )
                text_ok += 1
            except Exception as e:
                tqdm.write(f"⚠️  Text embed failed {reel_id}: {e}")
                text_fail += 1

    # Step 4: image embedding (skip if already indexed)
    if not already_has_image and thumb_exists:
        img_embed = get_clip_embedding(thumb_path)
        if img_embed:
            try:
                image_collection.upsert(
                    ids=[reel_id], embeddings=[img_embed], metadatas=[metadata]
                )
                img_ok += 1
            except Exception as e:
                tqdm.write(f"⚠️  Image embed failed {reel_id}: {e}")
                img_fail += 1
        else:
            img_fail += 1

# ── Save updated processed JSON to Drive ─────────────────────────────────────
with open(PROCESSED_JSON, "w", encoding="utf-8") as f:
    json.dump(deduped, f, indent=2, ensure_ascii=False)

print(f"""
✅ ChromaDB populated!
   Text  → {text_ok} stored  {text_fail} failed
   Image → {img_ok} stored  {img_fail} failed
   Skipped (already indexed): {skipped}
   Text collection  : {text_collection.count()} docs
   Image collection : {image_collection.count()} docs
   Saved → {PROCESSED_JSON}
""")

In [ ]:
image_col = client.get_or_create_collection(
    name="reel_images", metadata={"hnsw:space": "cosine"})

os.makedirs("thumbnails", exist_ok=True)
ok = fail = skip = 0

for reel in tqdm(deduped, desc="CLIP embedding", unit="reel"):  # ← uses normalized data
    reel_id   = reel["id"]                      # ← already "ig_..." from normalize_instagram
    thumb_url = reel.get("thumbnail", "")       # ← normalized field name

    if not thumb_url:
        skip += 1
        continue

    thumb_path = f"thumbnails/{reel_id}.jpg"
    if not os.path.exists(thumb_path):
        if not download_thumbnail(thumb_url, thumb_path):  # ← reuses helper from cell 1
            skip += 1
            continue

    embed = get_clip_embedding(thumb_path)       # ← reuses helper from cell 1
    if not embed:
        fail += 1
        continue

    try:
        metadata = {
            "video_url": reel.get("video_url", ""),
            "owner":     reel.get("owner", ""),
            "likes":     reel.get("likes", 0),
            "comments":  reel.get("comments", 0),
            "duration":  reel.get("duration", 0),
            "source":    reel.get("source", "instagram"),
            "caption":   reel.get("caption", "")[:500],
        }
        image_col.upsert(ids=[reel_id], embeddings=[embed], metadatas=[metadata])
        ok += 1
    except Exception as e:
        tqdm.write(f"CLIP failed {reel_id}: {e}")
        fail += 1

print(f"""
Done!
   Embedded : {ok:,}
   Failed   : {fail:,}
   Skipped  : {skip:,}
   Image collection: {image_col.count():,} docs
""")

In [ ]:
print(f"Total records   : {len(deduped)}")
print(f"Thumbnails on disk: {len(os.listdir('thumbnails'))}")
patched = sum(1 for r in deduped if r.get("thumb_path"))
print(f"Patched: {patched}")

In [ ]:
def export_to_pkl(collection_name: str, output_path: str):
    col   = client.get_collection(collection_name)   # ← uses existing client
    total = col.count()
    print(f"Exporting {collection_name}: {total:,} vectors...")

    BATCH = 500
    all_ids = all_embeddings = all_documents = all_metadatas = []
    all_ids, all_embeddings, all_documents, all_metadatas = [], [], [], []

    for offset in tqdm(range(0, total, BATCH), desc=f"Pulling {collection_name}"):
        result = col.get(
            limit=BATCH, offset=offset,
            include=["embeddings", "documents", "metadatas"]
        )
        all_ids.extend(result["ids"])
        all_embeddings.extend(result["embeddings"])
        all_metadatas.extend(result["metadatas"])
        if result.get("documents"):
            all_documents.extend(result["documents"])

    payload = {
        "ids": all_ids, "embeddings": all_embeddings,
        "metadatas": all_metadatas, "documents": all_documents,
        "count": len(all_ids), "collection": collection_name,
    }

    with open(output_path, "wb") as f:
        pickle.dump(payload, f)

    size = os.path.getsize(output_path) / 1e6
    print(f"Saved {output_path} — {len(all_ids):,} vectors — {size:.1f} MB\n")
    return payload

text_data  = export_to_pkl("reel_text",   "text_embeddings.pkl")
image_data = export_to_pkl("reel_images", "image_embeddings.pkl")

print("Verification:")
print(f"   text_embeddings.pkl  -> {text_data['count']:,} vectors, dim={len(text_data['embeddings'][0])}")
print(f"   image_embeddings.pkl -> {image_data['count']:,} vectors, dim={len(image_data['embeddings'][0])}")

NameError: name 'client' is not defined

In [ ]:
# ── Cell 1: PIPELINE SETUP ───────────────────────────────────────────────────
# Loads ChromaDB, reel_lookup, pkl embeddings, and all search functions
# (build_where_filter, text_search, image_text_search, hybrid_search,
#  deduplicate, query_reels) — sourced from pipeline.py, no redefinition here
TEXT_EMBEDDINGS_PKL="/content/drive/MyDrive/Dataset-json/text_embeddings.pkl"
_client          = chromadb.PersistentClient(path=CHROMA_DIR)
text_collection  = _client.get_or_create_collection("reel_text",   metadata={"hnsw:space": "cosine"})
image_collection = _client.get_or_create_collection("reel_images", metadata={"hnsw:space": "cosine"})
print(f"[pipeline] Text: {text_collection.count()} | Image: {image_collection.count()}")

with open(PROCESSED_JSON, encoding="utf-8") as f:
    _reels = json.load(f)
reel_lookup = {str(r["id"]): r for r in _reels}
print(f"[pipeline] Loaded {len(_reels)} reels")

def _load_pkl(path: str, label: str) -> dict:
    if not os.path.exists(path):
        print(f"[pipeline] ⚠️  {label} pkl not found — dedup skipped")
        return {}
    with open(path, "rb") as f:
        data = pickle.load(f)
    if isinstance(data, dict):
        return {str(k): np.array(v) for k, v in data.items()}
    if isinstance(data, (list, np.ndarray)):
        arr     = np.array(data)
        id_list = [str(r["id"]) for r in _reels]
        n       = min(len(arr), len(id_list))
        return {id_list[i]: arr[i] for i in range(n)}
    return {}

text_embeds_pkl  = _load_pkl(TEXT_EMBEDDINGS_PKL,  "text")
# image_embeds_pkl = _load_pkl(IMAGE_EMBEDDINGS_PKL, "image")
print(f"[pipeline] Text pkl: {len(text_embeds_pkl)} ")

[pipeline] Text: 1139 | Image: 0
[pipeline] Loaded 1140 reels
[pipeline] Text pkl: 6 


In [ ]:
# ── Cell 2: LLM + SCHEMAS + PROMPT ───────────────────────────────────────────
from langchain_aws import ChatBedrock  # ← Bedrock instead of Groq
os.environ["AWS_ACCESS_KEY_ID"] = "ENTER YOUR ACCESS KEY CREDENTIALS"
os.environ["AWS_SECRET_ACCESS_KEY"] = "ENTER YOUR ACCESS KEY CREDENTIALS"
os.environ["AWS_DEFAULT_REGION"] = "ap-south-1"

_base_llm = ChatBedrock(
    model_id="meta.llama3-70b-instruct-v1:0",
    region_name="ap-south-1",
    model_kwargs={"max_gen_len": 2048},
)

# ── Pydantic schemas ──────────────────────────────────────────────────────────
class IdeaStructure(BaseModel):
    concept:       str
    hook:          str
    structure:     List[str]
    emotion:       str
    why_it_works:  str
    reference_url: str

class OptimizationVariant(BaseModel):
    change: str
    add:    str
    result: str

class OptimizationSuggestion(BaseModel):
    second_idea_emotional_variant: OptimizationVariant

class BestFitRecommendation(BaseModel):
    best_idea_index: int
    reason:          str

class AnalysisBlock(BaseModel):
    performance_drivers: List[str]
    engagement_triggers: List[str]

class StrategistOutput(BaseModel):
    analysis:                AnalysisBlock
    patterns:                List[str]
    ideas:                   List[IdeaStructure]
    best_fit_recommendation: BestFitRecommendation
    optimization_suggestion: OptimizationSuggestion

# ── System prompt ─────────────────────────────────────────────────────────────
system_prompt = """
You are an expert social media performance analyst and viral content strategist specializing in Instagram Reels.
You analyze high-performing reels to uncover repeatable success patterns and transform them into highly actionable viral content ideas.

Focus on:
- audience psychology  • scroll-stopping hooks  • retention mechanics
- emotional triggers   • relatability & shareability

Avoid generic observations.

You are given REAL reels retrieved from a database. Each reel may include:
- captions, transcript excerpts, engagement metrics, creator handles

━━━━━━━━━━━━━━━━━━━━
YOUR TASK
━━━━━━━━━━━━━━━━━━━━
1️⃣ Analyze why the retrieved reels performed well (hook effectiveness, curiosity gaps,
   pacing, emotional triggers, novelty vs familiarity). Be specific and concise.

2️⃣ Identify repeating patterns (hook formats, storytelling flow, tone, pain points,
   visual framing, engagement triggers). Prioritize: repeatable, psychologically
   compelling, niche-relevant patterns.

3️⃣ Generate 3 HIGHLY SPECIFIC viral reel ideas from those patterns.
   Native to the niche, optimized for retention and shares. No generic trends.

━━━━━━━━━━━━━━━━━━━━
QUALITY RULES
━━━━━━━━━━━━━━━━━━━━
- Specific not generic         • Optimize retention & shareability
- Hooks must create curiosity gaps    • Ideas must be immediately executable

━━━━━━━━━━━━━━━━━━━━
STRICT OUTPUT FORMAT
━━━━━━━━━━━━━━━━━━━━
Respond with ONLY a valid JSON object — no markdown, no explanation, no extra text.

{{
  "analysis": {{
    "performance_drivers": ["...", "..."],
    "engagement_triggers": ["...", "..."]
  }},
  "patterns": ["...", "..."],
  "ideas": [
    {{
      "concept": "...", "hook": "...",
      "structure": ["step 1", "step 2", "step 3", "step 4", "step 5"],
      "emotion": "...", "why_it_works": "...", "reference_url": "..."
    }}
  ],
  "best_fit_recommendation": {{"best_idea_index": 0, "reason": "..."}},
  "optimization_suggestion": {{
    "second_idea_emotional_variant": {{"change": "...", "add": "...", "result": "..."}}
  }}
}}

Retrieved Reels:
{context}
"""

In [ ]:
import re
# ── Cell 3: LANGGRAPH NODES + GRAPH ──────────────────────────────────────────
# HybridReelRetriever and conversational_rag dropped — LangGraph only
from langgraph.graph import StateGraph, END
# ── query_reels ───────────────────────────────────────────────────────────────
def query_reels(
    query:       str,
    top_k:       int   = 8,
    text_weight: float = 0.6,
) -> list:
    image_weight = 1.0 - text_weight
    scores = defaultdict(lambda: {"text": 0.0, "image": 0.0})

    # Text retrieval
    try:
        text_embed   = text_model.encode(query, normalize_embeddings=True).tolist()
        text_results = text_collection.query(
            query_embeddings=[text_embed],
            n_results=min(top_k, text_collection.count()),
            include=["distances", "metadatas", "documents"]
        )
        for i, doc_id in enumerate(text_results["ids"][0]):
            scores[doc_id]["text"]     = 1.0 - text_results["distances"][0][i]
            scores[doc_id]["metadata"] = text_results["metadatas"][0][i]
            scores[doc_id]["document"] = (text_results["documents"][0][i]
                                          if text_results.get("documents") else "")
    except Exception as e:
        print(f"⚠️  Text retrieval failed: {e}")

    # Image retrieval (skipped if collection empty)
    if image_collection.count() > 0:
        try:
            image_results = image_collection.query(
                query_embeddings=[text_embed],
                n_results=min(top_k, image_collection.count()),
                include=["distances", "metadatas"]
            )
            for i, doc_id in enumerate(image_results["ids"][0]):
                scores[doc_id]["image"] = 1.0 - image_results["distances"][0][i]
                if "metadata" not in scores[doc_id]:
                    scores[doc_id]["metadata"] = image_results["metadatas"][0][i]
        except Exception as e:
            print(f"⚠️  Image retrieval failed: {e}")

    # Blend + attach full reel data
    results = []
    for doc_id, s in scores.items():
        blended = text_weight * s["text"] + image_weight * s["image"]
        results.append({
            "id":       doc_id,
            "score":    round(blended, 5),
            "metadata": s.get("metadata", {}),
            "document": s.get("document", ""),
            "reel":     reel_lookup.get(doc_id, {}),
        })

    results.sort(key=lambda x: x["score"], reverse=True)
    return results[:top_k]

print("✅ query_reels ready")
class PipelineState(TypedDict):
    query:                  str
    expanded_query:         str
    retrieved_docs:         List[Any]
    reranked_docs:          List[Any]
    strategist_output:      Optional[Any]
    eval_passed:            bool
    retry_count:            int
    final_output:           Optional[dict]
    alpha:                  float
    conversational_answer:  Optional[str]
    prev_strategist_output: Optional[Any]

MAX_RETRIES          = 2
TOP_K                = 8
DEFAULT_ALPHA        = 0.4
MIN_IDEAS_NEEDED     = 2
MIN_IDEA_CONCEPT_LEN = 10

chat_history = []  # ← shared across session turns

def retrieve_node(state: PipelineState) -> PipelineState:
    query = state["expanded_query"] if state["retry_count"] > 0 else state["query"]
    print(f"\n[retrieve] '{query}' | Retry #{state['retry_count']}")
    return {**state, "retrieved_docs": query_reels(query, top_k=TOP_K * 3, text_weight=0.6)}

def cosine_rerank_node(state: PipelineState) -> PipelineState:
    docs  = state["retrieved_docs"]
    query = state["expanded_query"] if state["retry_count"] > 0 else state["query"]
    alpha = state.get("alpha", DEFAULT_ALPHA)

    if not docs:
        return {**state, "reranked_docs": docs}

    print(f"[rerank] {len(docs)} docs (alpha={alpha})")
    query_vec = text_model.encode(query, normalize_embeddings=True)

    reranked = []
    for doc in docs:
        meta      = doc.get("metadata", {})
        doc_text  = f"{meta.get('caption', '')} {doc.get('reel', {}).get('transcript', '')}".strip()
        cosine    = float(np.dot(query_vec, text_model.encode(doc_text, normalize_embeddings=True))) if doc_text else 0.0
        blended   = alpha * cosine + (1 - alpha) * doc.get("score", 0.0)
        reranked.append({**doc, "blended_score": round(blended, 5)})

    reranked.sort(key=lambda x: x["blended_score"], reverse=True)
    reranked = reranked[:TOP_K]
    print(f"[rerank] Top: {reranked[0]['blended_score']} | Bottom: {reranked[-1]['blended_score']}")
    return {**state, "reranked_docs": reranked}


CONVERSATIONAL_KEYWORDS = [
    "which idea", "make the", "more emotional", "less funny",
    "more funny", "change the", "adjust", "modify", "rewrite",
    "works best", "better for", "suggest", "what about",
]

def is_conversational(query: str) -> bool:
    q = query.lower()
    return any(kw in q for kw in CONVERSATIONAL_KEYWORDS)

def extract_json_from_raw(raw: str) -> str:
    """Strip prose preambles and extract the first JSON object."""
    # Try to find JSON block in ```json ... ``` fence first
    fenced = re.search(r"```json\s*(\{.*?\})\s*```", raw, re.DOTALL)
    if fenced:
        return fenced.group(1)
    # Fall back: find first { to last }
    start = raw.find("{")
    end   = raw.rfind("}")
    if start != -1 and end != -1 and end > start:
        return raw[start:end+1]
    return raw  # return as-is and let pydantic fail with a useful error

def strategist_node(state: PipelineState) -> PipelineState:
    docs  = state["reranked_docs"]
    query = state["query"]
    print(f"[strategist] Running on {len(docs)} docs...")

    # ── Conversational follow-up: answer in prose, bypass schema ─────────────
    if is_conversational(query) and chat_history:
        print("[strategist] Detected follow-up — conversational mode")
        raw = _base_llm.invoke([
            ("system",
             "You are a social media strategist. Answer the user's follow-up question "
             "about the previously generated reel ideas. Be specific, direct, and concise. "
             "No JSON needed — just answer clearly in plain text."
            ),
            *[(m.type, m.content) for m in chat_history],
            ("human", query),
        ]).content.strip()

        chat_history.append(HumanMessage(content=query))
        chat_history.append(AIMessage(content=raw))

        # Carry previous strategist output forward unchanged
        return {
            **state,
            "conversational_answer":  raw,
            "strategist_output":      state.get("prev_strategist_output"),
            "eval_passed":            True,   # skip evaluator for conversational
        }

    # ── Standard retrieval-based mode ────────────────────────────────────────
    context = "\n\n---\n\n".join([
        f"Owner: @{d['metadata'].get('owner','?')} | "
        f"Likes: {d['metadata'].get('likes', 0):,} | "
        f"Duration: {d['metadata'].get('duration', 0)}s | "
        f"URL: {d['metadata'].get('embed_url', d['metadata'].get('video_url',''))}\n"
        f"Caption: {d['metadata'].get('caption', '')}\n"
        f"Transcript: {d.get('reel', {}).get('transcript', '')}"
        for d in docs
    ])

    retry_nudge = (
        "\n\n⚠️ RETRY: Previous output was too generic. "
        "Use concrete details from the retrieved reels."
    ) if state["retry_count"] > 0 else ""

    filled_prompt = (system_prompt
        .replace("{context}", context)
        .replace("{{", "{")
        .replace("}}", "}")) + retry_nudge

    messages  = [("system", filled_prompt)]
    messages += [(m.type, m.content) for m in chat_history]
    messages += [("human", query)]

    time.sleep(5)
    raw = _base_llm.invoke(messages).content

    try:
        clean  = extract_json_from_raw(raw)
        parsed = StrategistOutput.model_validate_json(clean)
    except Exception as e:
        print(f"[strategist] Parse error: {e}")
        print(f"[strategist] Raw sample: {raw[:300]}")
        parsed = None

    chat_history.append(HumanMessage(content=query))
    chat_history.append(AIMessage(content=raw))
    return {
        **state,
        "strategist_output":      parsed,
        "prev_strategist_output": parsed or state.get("prev_strategist_output"),
        "conversational_answer":  None,
    }

def evaluator_node(state: PipelineState) -> PipelineState:
    output = state["strategist_output"]
    if output is None:
        print("[evaluator] ❌ No parsed output")
        return {**state, "eval_passed": False}

    ideas       = output.ideas
    short_hooks = sum(1 for i in ideas if len(i.concept) < MIN_IDEA_CONCEPT_LEN)
    passed      = len(ideas) >= MIN_IDEAS_NEEDED and short_hooks == 0 and output.best_fit_recommendation is not None

    print(f"[evaluator] {'✅ Passed' if passed else '❌ Failed'} ({len(ideas)} ideas)")
    return {**state, "eval_passed": passed}

def query_expander_node(state: PipelineState) -> PipelineState:
    print("[expander] Broadening query...")
    expanded = _base_llm.invoke([("human",
        f"The query '{state['query']}' returned low-quality results. "
        "Rewrite it as a slightly broader version (5-10 words max). Return ONLY the new query."
    )]).content.strip().strip('"').strip("'")
    print(f"[expander] '{state['query']}' → '{expanded}'")
    return {**state, "expanded_query": expanded, "retry_count": state["retry_count"] + 1}

def finalizer_node(state: PipelineState) -> PipelineState:
    output = state["strategist_output"]
    docs   = state["reranked_docs"]
    final  = {
        "query":                state["query"],
        "retries_used":         state["retry_count"],
        "alpha_used":           state.get("alpha", DEFAULT_ALPHA),
        "answer":               output.model_dump() if output else None,
        "conversational_answer": state.get("conversational_answer"),  # ← new
        "sources": [{
            "owner":    d["metadata"].get("owner", "?"),
            "likes":    d["metadata"].get("likes", 0),
            "duration": d["metadata"].get("duration", 0),
            "score":    d.get("blended_score", d.get("score", 0)),
            "url":      d["metadata"].get("embed_url") or d["metadata"].get("video_url", ""),
        } for d in docs],
    }
    print(f"[final] Done. Retries: {state['retry_count']}")
    return {**state, "final_output": final}

def route_after_eval(state: PipelineState) -> str:
    if state["eval_passed"]:                    return "finalize"
    if state["retry_count"] < MAX_RETRIES:      return "expand_and_retry"
    print("[router] Max retries reached — finalizing anyway")
    return "finalize"

def build_graph():
    g = StateGraph(PipelineState)
    g.add_node("retrieve",   retrieve_node)
    g.add_node("rerank",     cosine_rerank_node)
    g.add_node("strategist", strategist_node)
    g.add_node("evaluator",  evaluator_node)
    g.add_node("expander",   query_expander_node)
    g.add_node("finalize",   finalizer_node)

    g.set_entry_point("retrieve")
    g.add_edge("retrieve",   "rerank")
    g.add_edge("rerank",     "strategist")
    g.add_edge("strategist", "evaluator")
    g.add_conditional_edges("evaluator", route_after_eval,
        {"finalize": "finalize", "expand_and_retry": "expander"})
    g.add_edge("expander", "retrieve")
    g.add_edge("finalize", END)
    return g.compile()

pipeline_graph = build_graph()
print("[graph] Pipeline compiled ✅")

✅ query_reels ready
[graph] Pipeline compiled ✅


In [ ]:
# test case thing

# ── MODELS ────────────────────────────────────────────────────────────────────
CLIP_MODEL_ID = "openai/clip-vit-base-patch32"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device.upper()}")

print("Loading models...")
text_model     = SentenceTransformer("all-MiniLM-L6-v2", device=device)
clip_model     = CLIPModel.from_pretrained(CLIP_MODEL_ID).to(device).eval()
clip_processor = CLIPProcessor.from_pretrained(CLIP_MODEL_ID)
whisper_model  = whisper.load_model("medium", device=device)
print("✅ All models loaded!")

Device: CPU
Loading models...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ All models loaded!


In [ ]:
def run_optimized_pipeline(
    query: str,
    alpha: float = DEFAULT_ALPHA,
    prev_strategist_output=None
) -> dict:
    final_state = pipeline_graph.invoke({
        "query":                  query,
        "expanded_query":         query,
        "retrieved_docs":         [],
        "reranked_docs":          [],
        "strategist_output":      None,
        "eval_passed":            False,
        "retry_count":            0,
        "final_output":           None,
        "alpha":                  alpha,
        "conversational_answer":  None,
        "prev_strategist_output": prev_strategist_output,  # ← carry forward
    })
    result = final_state["final_output"]

    print(f"\n👤 Query: {result['query']}")
    if result.get("conversational_answer"):
        print(f"\n🤖 Response:\n{result['conversational_answer']}")  # ← prose for follow-ups
    else:
        print(f"\n🤖 Strategist Output:")
        print(json.dumps(result["answer"], indent=2, ensure_ascii=False))
    print(f"\n📦 Sources ({len(result['sources'])}):")
    for s in result["sources"]:
        print(f"  @{s['owner']} | ❤️ {s['likes']:,} | ⏱ {s['duration']}s | score: {s['score']} | {s['url']}")
    print(f"\n⚙️  Alpha: {result['alpha_used']} | Retries: {result['retries_used']}")
    print("\n" + "=" * 70)
    return final_state["final_output"], final_state.get("prev_strategist_output")

def run_optimized_session(queries: list, alpha: float = DEFAULT_ALPHA):
    chat_history.clear()
    print("\n💬 Starting optimized strategist session...\n")
    results = []
    prev_strategist_output = None
    for q in queries:
        result, prev_strategist_output = run_optimized_pipeline(
            q, alpha=alpha, prev_strategist_output=prev_strategist_output
        )
        results.append(result)
    return results

run_optimized_session([
    "funny relatable student struggles",
    "make the second idea more emotional and less funny",
    "which idea works best for a college page with 10k followers?",
])


💬 Starting optimized strategist session...


[retrieve] 'funny relatable student struggles' | Retry #0
[rerank] 24 docs (alpha=0.4)
[rerank] Top: 0.39093 | Bottom: 0.32185
[strategist] Running on 8 docs...
[evaluator] ✅ Passed (3 ideas)
[final] Done. Retries: 0

👤 Query: funny relatable student struggles

🤖 Strategist Output:
{
  "analysis": {
    "performance_drivers": [
      "relatable content",
      "humor",
      "nostalgia"
    ],
    "engagement_triggers": [
      "meme culture",
      "college life references",
      "emotional connection"
    ]
  },
  "patterns": [
    "meme page format",
    "humorous storytelling",
    "college life stereotypes"
  ],
  "ideas": [
    {
      "concept": "College Life Struggles",
      "hook": "When you finally understand the assignment",
      "structure": [
        "intro with a relatable meme",
        "series of humorous skits",
        "conclusion with a funny twist"
      ],
      "emotion": "laughter",
      "why_it_works": "relatable

[{'query': 'funny relatable student struggles',
  'retries_used': 0,
  'alpha_used': 0.4,
  'answer': {'analysis': {'performance_drivers': ['relatable content',
     'humor',
     'nostalgia'],
    'engagement_triggers': ['meme culture',
     'college life references',
     'emotional connection']},
   'patterns': ['meme page format',
    'humorous storytelling',
    'college life stereotypes'],
   'ideas': [{'concept': 'College Life Struggles',
     'hook': 'When you finally understand the assignment',
     'structure': ['intro with a relatable meme',
      'series of humorous skits',
      'conclusion with a funny twist'],
     'emotion': 'laughter',
     'why_it_works': 'relatable content and humor create a strong emotional connection with the audience',
     'reference_url': 'https://hackathon-reels-ayush-2026.s3.amazonaws.com/reels/DUNmMPkAR3W.mp4'},
    {'concept': 'Types of College Students',
     'hook': "When you're the lazy bum in class",
     'structure': ['intro with a humo